
# Laboratorio interactivo: potencial de acción con Hodgkin–Huxley

Este notebook está pensado para **VS Code** y reúne:

- simulación del modelo de Hodgkin–Huxley,
- interpretación física,
- análisis dinámico,
- visualizaciones interactivas con **sliders** y **botones**,
- mecanismos para **reducir flickering** en las gráficas.

## Estructura

1. **Umbral de excitación**  
2. **Período refractario**  
3. **Rol de cada canal**

## Recomendaciones para VS Code

1. Abre este archivo con el editor de notebooks.
2. Usa un entorno con `numpy`, `matplotlib`, `ipywidgets` e idealmente `ipympl`.
3. Si no tienes el backend interactivo, instala:

```bash
pip install numpy matplotlib ipywidgets ipympl
```

4. Ejecuta las celdas en orden.

> **Nota técnica sobre flickering:** en las secciones interactivas no se recrea la figura cada vez.  
> Se construye **una sola figura persistente** y luego solo se actualizan los artistas (`Line2D`, textos y marcadores) usando `set_data(...)` y `draw_idle()`.  
> Además, los sliders se declaran con `continuous_update=True` para evitar redibujos excesivos mientras arrastras el control.



## Modelo

Usaremos la forma clásica del modelo de Hodgkin–Huxley para el potencial de membrana \(V(t)\):

$$
C_m \frac{dV}{dt}
=
I_{\mathrm{ext}}(t)
-
I_{Na}
-
I_K
-
I_L,
$$

con

$$
I_{Na} = \bar g_{Na} m^3 h (V - E_{Na}), \qquad
I_K = \bar g_K n^4 (V - E_K), \qquad
I_L = g_L (V - E_L).
$$

Las variables de compuerta \(m,h,n\) obedecen

$$
\frac{dx}{dt} = \alpha_x(V)(1-x) - \beta_x(V)x, \qquad x \in \{m,h,n\}.
$$

En el artículo original, Hodgkin y Huxley muestran que la corriente total puede descomponerse en una corriente capacitiva más corrientes iónicas, y que las conductancias efectivas quedan modeladas como

$$
g_K = \bar g_K\, n^4,
\qquad
g_{Na} = \bar g_{Na}\, m^3 h.
$$

La idea física es:

- \(m\): **activación rápida** del sodio,
- \(h\): **inactivación** del sodio,
- \(n\): **activación lenta** del potasio.

Ese desacople temporal es precisamente lo que permite el pico rápido del spike y la repolarización posterior.



---

# 1. Umbral de excitación

## Interpretación física

El potencial de acción no aparece para cualquier corriente.  
Existe un **umbral** a partir del cual la activación rápida del sodio supera a las corrientes estabilizantes y el sistema entra en un régimen regenerativo.

En lenguaje del modelo:

- si el estímulo es pequeño, la membrana se despolariza y luego regresa;
- si el estímulo supera el umbral, el sodio entra masivamente y aparece el spike;
- después, el potasio y la inactivación del sodio detienen la subida y repolarizan.

## Qué mirar aquí

- La gráfica \(V(t)\): si hay o no spike.
- Las compuertas \(m,h,n\): quién lidera la dinámica.
- Las corrientes iónicas: el sodio domina la subida; el potasio domina la bajada.
- El plano \((V,n)\): sirve como una reducción geométrica para visualizar el cruce del umbral.

> En el plano \((V,n)\), la trayectoria cambia radicalmente cuando la corriente empuja al sistema más allá de la región estable.


In [ ]:
# =======================================
# Sección 1: Umbral de excitación
# Versión corregida para VS Code
# =======================================

%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# -------------------------------------------------
# Configuración visual
# -------------------------------------------------
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 10

# Muy importante cuando se incrusta fig.canvas en widgets
plt.ioff()

# -------------------------------------------------
# Parámetros base
# -------------------------------------------------
PARAMS_BASE = {
    "Cm": 1.0,      # uF/cm^2
    "gNa": 120.0,   # mS/cm^2
    "gK": 36.0,     # mS/cm^2
    "gL": 0.3,      # mS/cm^2
    "ENa": 50.0,    # mV
    "EK": -77.0,    # mV
    "EL": -54.4,    # mV
    "V_rest": -65.0 # mV
}

# -------------------------------------------------
# Funciones auxiliares sin warnings numéricos
# -------------------------------------------------
def _as_array(x):
    arr = np.asarray(x, dtype=float)
    scalar = arr.ndim == 0
    if scalar:
        arr = arr.reshape(1)
    return arr, scalar

def alpha_n(V):
    x, scalar = _as_array(V)
    z = x + 55.0
    out = np.empty_like(z)
    mask = np.abs(z) < 1e-8
    out[mask] = 0.1
    out[~mask] = 0.01 * z[~mask] / (1.0 - np.exp(-z[~mask] / 10.0))
    return float(out[0]) if scalar else out

def beta_n(V):
    x, scalar = _as_array(V)
    out = 0.125 * np.exp(-(x + 65.0) / 80.0)
    return float(out[0]) if scalar else out

def alpha_m(V):
    x, scalar = _as_array(V)
    z = x + 40.0
    out = np.empty_like(z)
    mask = np.abs(z) < 1e-8
    out[mask] = 1.0
    out[~mask] = 0.1 * z[~mask] / (1.0 - np.exp(-z[~mask] / 10.0))
    return float(out[0]) if scalar else out

def beta_m(V):
    x, scalar = _as_array(V)
    out = 4.0 * np.exp(-(x + 65.0) / 18.0)
    return float(out[0]) if scalar else out

def alpha_h(V):
    x, scalar = _as_array(V)
    out = 0.07 * np.exp(-(x + 65.0) / 20.0)
    return float(out[0]) if scalar else out

def beta_h(V):
    x, scalar = _as_array(V)
    out = 1.0 / (1.0 + np.exp(-(x + 35.0) / 10.0))
    return float(out[0]) if scalar else out

def x_inf(alpha, beta, V):
    a = alpha(V)
    b = beta(V)
    return a / (a + b)

def tau_x(alpha, beta, V):
    a = alpha(V)
    b = beta(V)
    return 1.0 / (a + b)

def m_inf(V): return x_inf(alpha_m, beta_m, V)
def h_inf(V): return x_inf(alpha_h, beta_h, V)
def n_inf(V): return x_inf(alpha_n, beta_n, V)

def tau_m(V): return tau_x(alpha_m, beta_m, V)
def tau_h(V): return tau_x(alpha_h, beta_h, V)
def tau_n(V): return tau_x(alpha_n, beta_n, V)

def current_from_pulses(t, pulses):
    I = np.zeros_like(t, dtype=float)
    for t0, dur, amp in pulses:
        I[(t >= t0) & (t < t0 + dur)] += amp
    return I

def initial_gates(V0):
    return float(m_inf(V0)), float(h_inf(V0)), float(n_inf(V0))

def hh_rhs(state, Iext, params):
    V, m, h, n = state
    Cm  = params["Cm"]
    gNa = params["gNa"]
    gK  = params["gK"]
    gL  = params["gL"]
    ENa = params["ENa"]
    EK  = params["EK"]
    EL  = params["EL"]

    gNa_t = gNa * (m**3) * h
    gK_t  = gK * (n**4)

    INa = gNa_t * (V - ENa)
    IK  = gK_t  * (V - EK)
    IL  = gL    * (V - EL)

    dV = (Iext - INa - IK - IL) / Cm
    dm = alpha_m(V) * (1.0 - m) - beta_m(V) * m
    dh = alpha_h(V) * (1.0 - h) - beta_h(V) * h
    dn = alpha_n(V) * (1.0 - n) - beta_n(V) * n
    return np.array([dV, dm, dh, dn], dtype=float)

def simulate_hh(
    pulses,
    t_max=40.0,
    dt=0.01,
    params=None,
    gNa_scale=1.0,
    gK_scale=1.0,
    gL_scale=1.0,
    V0=None,
):
    if params is None:
        params = PARAMS_BASE.copy()
    else:
        params = params.copy()

    params["gNa"] *= gNa_scale
    params["gK"]  *= gK_scale
    params["gL"]  *= gL_scale

    if V0 is None:
        V0 = params["V_rest"]

    t = np.arange(0.0, t_max + dt, dt)
    I = current_from_pulses(t, pulses)

    V = np.empty_like(t)
    m = np.empty_like(t)
    h = np.empty_like(t)
    n = np.empty_like(t)

    V[0] = V0
    m[0], h[0], n[0] = initial_gates(V0)

    for k in range(len(t) - 1):
        y = np.array([V[k], m[k], h[k], n[k]], dtype=float)

        k1 = hh_rhs(y, I[k], params)
        k2 = hh_rhs(y + 0.5 * dt * k1, I[k], params)
        k3 = hh_rhs(y + 0.5 * dt * k2, I[k], params)
        k4 = hh_rhs(y + dt * k3, I[k + 1], params)

        y_next = y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

        V[k+1] = y_next[0]
        m[k+1] = np.clip(y_next[1], 0.0, 1.0)
        h[k+1] = np.clip(y_next[2], 0.0, 1.0)
        n[k+1] = np.clip(y_next[3], 0.0, 1.0)

    gNa_t = params["gNa"] * (m**3) * h
    gK_t  = params["gK"]  * (n**4)
    gL_t  = params["gL"] * np.ones_like(t)

    INa = gNa_t * (V - params["ENa"])
    IK  = gK_t  * (V - params["EK"])
    IL  = gL_t  * (V - params["EL"])

    return {
        "t": t,
        "I": I,
        "V": V,
        "m": m,
        "h": h,
        "n": n,
        "INa": INa,
        "IK": IK,
        "IL": IL,
        "gNa_t": gNa_t,
        "gK_t": gK_t,
        "gL_t": gL_t,
        "params": params,
    }

def spike_summary(data, t_window_start=0.0):
    t = data["t"]
    V = data["V"]
    mask = t >= t_window_start
    if not np.any(mask):
        return {"peak_V": np.nan, "peak_t": np.nan, "spike": False}
    idx_local = np.argmax(V[mask])
    idx = np.where(mask)[0][0] + idx_local
    peak_V = V[idx]
    peak_t = t[idx]
    spike = peak_V > 0.0
    return {"peak_V": float(peak_V), "peak_t": float(peak_t), "spike": bool(spike)}

def vn_nullclines(Iext, params, V_grid):
    n_nc = n_inf(V_grid)

    gNa = params["gNa"]
    gK  = params["gK"]
    gL  = params["gL"]
    ENa = params["ENa"]
    EK  = params["EK"]
    EL  = params["EL"]

    numerator = (
        Iext
        - gNa * (m_inf(V_grid)**3) * h_inf(V_grid) * (V_grid - ENa)
        - gL * (V_grid - EL)
    )
    denominator = gK * (V_grid - EK)

    with np.errstate(invalid="ignore", divide="ignore"):
        value = numerator / denominator
        n_v = np.where(value >= 0.0, np.power(value, 0.25), np.nan)

    return n_nc, n_v

def setup_axes_style(ax, xlabel=None, ylabel=None, title=None, xlim=None, ylim=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)

# -------------------------------------------------
# Widgets
# -------------------------------------------------
V_GRID = np.linspace(-90, 60, 400)

amp_slider_1 = widgets.FloatSlider(
    value=8.0, min=0.0, max=20.0, step=0.5,
    description="I (uA/cm²)", continuous_update=True,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

start_slider_1 = widgets.FloatSlider(
    value=10.0, min=2.0, max=20.0, step=0.5,
    description="inicio (ms)", continuous_update=True,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

dur_slider_1 = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description="duración (ms)", continuous_update=True,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

ui_1 = widgets.VBox([
    widgets.HTML("<h3>1. Umbral de excitación</h3>"),
    widgets.HTML(
        "<p>Mueve la amplitud, inicio y duración del pulso. "
        "La figura se actualiza sin recrearse para reducir flickering.</p>"
    ),
    widgets.HBox([amp_slider_1, start_slider_1, dur_slider_1])
])

# -------------------------------------------------
# Figura persistente
# -------------------------------------------------
fig1, axs1 = plt.subplots(2, 2, figsize=(11, 8))
ax1a, ax1b, ax1c, ax1d = axs1.ravel()

setup_axes_style(
    ax1a,
    xlabel="tiempo (ms)", ylabel="V (mV)",
    title="Potencial de membrana",
    xlim=(0, 40), ylim=(-90, 60)
)
setup_axes_style(
    ax1b,
    xlabel="tiempo (ms)", ylabel="compuertas",
    title="Variables de compuerta",
    xlim=(0, 40), ylim=(0, 1)
)
setup_axes_style(
    ax1c,
    xlabel="tiempo (ms)", ylabel="corriente (uA/cm²)",
    title="Corrientes iónicas y estímulo",
    xlim=(0, 40), ylim=(-250, 250)
)
setup_axes_style(
    ax1d,
    xlabel="V (mV)", ylabel="n",
    title="Plano reducido (V, n)",
    xlim=(-90, 60), ylim=(0, 1)
)

line1_V, = ax1a.plot([], [], lw=2, label="V(t)")
vline1_start = ax1a.axvline(10.0, color="gray", ls="--", alpha=0.6)
vline1_end   = ax1a.axvline(11.0, color="gray", ls="--", alpha=0.6)
text1 = ax1a.text(0.02, 0.95, "", transform=ax1a.transAxes, va="top")

line1_m, = ax1b.plot([], [], lw=2, label="m")
line1_h, = ax1b.plot([], [], lw=2, label="h")
line1_n, = ax1b.plot([], [], lw=2, label="n")
ax1b.legend(loc="upper right")

line1_INa, = ax1c.plot([], [], lw=2, label="I_Na")
line1_IK,  = ax1c.plot([], [], lw=2, label="I_K")
line1_IL,  = ax1c.plot([], [], lw=2, label="I_L")
line1_Iext,= ax1c.plot([], [], lw=2, ls="--", label="I_ext")
ax1c.legend(loc="upper right")

line1_traj, = ax1d.plot([], [], lw=2, label="trayectoria")
line1_n_nc, = ax1d.plot([], [], lw=2, ls="--", label="dn/dt = 0")
line1_v_nc, = ax1d.plot([], [], lw=2, ls=":", label="dV/dt = 0 (aprox.)")
pt1_init, = ax1d.plot([], [], marker="o", ms=6, linestyle="None", label="inicio")
ax1d.legend(loc="best")

fig1.tight_layout()

# -------------------------------------------------
# Función de actualización
# -------------------------------------------------
def update_section_1(*_):
    amp = amp_slider_1.value
    t0 = start_slider_1.value
    dur = dur_slider_1.value

    data = simulate_hh([(t0, dur, amp)], t_max=40.0, dt=0.01)
    summary = spike_summary(data, t_window_start=t0)

    # series temporales
    line1_V.set_data(data["t"], data["V"])

    line1_m.set_data(data["t"], data["m"])
    line1_h.set_data(data["t"], data["h"])
    line1_n.set_data(data["t"], data["n"])

    line1_INa.set_data(data["t"], data["INa"])
    line1_IK.set_data(data["t"], data["IK"])
    line1_IL.set_data(data["t"], data["IL"])
    line1_Iext.set_data(data["t"], data["I"])

    # líneas verticales del estímulo
    vline1_start.set_xdata([t0, t0])
    vline1_end.set_xdata([t0 + dur, t0 + dur])

    # plano reducido
    n_nc, v_nc = vn_nullclines(amp, data["params"], V_GRID)
    line1_traj.set_data(data["V"], data["n"])
    line1_n_nc.set_data(V_GRID, n_nc)
    line1_v_nc.set_data(V_GRID, v_nc)
    pt1_init.set_data([data["V"][0]], [data["n"][0]])

    # texto resumen
    spike_text = "sí" if summary["spike"] else "no"
    text1.set_text(
        f"spike: {spike_text}\n"
        f"V pico = {summary['peak_V']:.2f} mV\n"
        f"t pico = {summary['peak_t']:.2f} ms"
    )

    fig1.canvas.draw_idle()

# -------------------------------------------------
# Conexión de eventos
# -------------------------------------------------
for w in (amp_slider_1, start_slider_1, dur_slider_1):
    w.observe(update_section_1, names="value")

# -------------------------------------------------
# Mostrar interfaz
# -------------------------------------------------
contenedor_1 = widgets.VBox([ui_1, fig1.canvas])
display(contenedor_1)

update_section_1()


### Lectura dinámica de esta sección

- Si aumentas la corriente y **no aparece spike**, estás todavía en el régimen subumbral.
- En el instante crítico, \(m\) crece muy rápido, mientras que \(h\) aún no ha caído del todo y \(n\) todavía no ha subido suficientemente.
- Esa diferencia de escalas temporales permite la **fase ascendente regenerativa**.
- Cuando \(n\) sube y \(h\) cae, el sistema abandona la región de ascenso y se repolariza.

Una forma útil de pensar el umbral es:

- el estímulo mueve el sistema en el espacio de estados;
- si no cruza la región inestable, vuelve al reposo;
- si la cruza, se dispara el spike.



---

# 2. Período refractario

## Interpretación física

Después de un potencial de acción, la membrana **no se comporta como si estuviera intacta**:

- \(h\) todavía está deprimida: el sodio no está completamente disponible;
- \(n\) todavía está elevada: el potasio sigue favoreciendo la salida de carga positiva.

Por eso, si aplicas un segundo pulso demasiado pronto:

- puede no producir nada (**refractario absoluto**),
- puede producir un spike pequeño o deformado (**refractario relativo**),
- o puede volver a producir un spike casi normal si la membrana ya recuperó sus variables internas.

## Qué mirar aquí

- La respuesta completa \(V(t)\).
- El zoom alrededor del segundo pulso.
- La recuperación de \(h\) y \(n\).
- La trayectoria en el plano \((n,h)\): ese plano muestra muy bien cómo se aleja y cómo regresa el sistema al reposo.


In [ ]:

# =======================================
# Sección 2: Período refractario
# =======================================

amp1_slider_2 = widgets.FloatSlider(
    value=10.0, min=6.0, max=20.0, step=0.5,
    description="I1", continuous_update=True
)
dur1_slider_2 = widgets.FloatSlider(
    value=1.0, min=0.1, max=4.0, step=0.1,
    description="dur1", continuous_update=True
)
delay_slider_2 = widgets.FloatSlider(
    value=12.0, min=1.0, max=30.0, step=0.5,
    description="retardo 2º", continuous_update=True
)
amp2_slider_2 = widgets.FloatSlider(
    value=10.0, min=0.0, max=25.0, step=0.5,
    description="I2", continuous_update=True
)
dur2_slider_2 = widgets.FloatSlider(
    value=1.0, min=0.1, max=4.0, step=0.1,
    description="dur2", continuous_update=True
)

ui_2 = widgets.VBox([
    widgets.HTML("<b>Período refractario</b>"),
    widgets.HBox([amp1_slider_2, dur1_slider_2, delay_slider_2]),
    widgets.HBox([amp2_slider_2, dur2_slider_2]),
])

fig2, axs2 = plt.subplots(2, 2, figsize=(11, 8))
ax2a, ax2b, ax2c, ax2d = axs2.ravel()

setup_axes_style(ax2a, xlabel="tiempo (ms)", ylabel="V (mV)", title="Respuesta completa", xlim=(0, 55), ylim=(-90, 60))
setup_axes_style(ax2b, xlabel="tiempo (ms)", ylabel="V (mV)", title="Zoom alrededor del segundo pulso", xlim=(8, 35), ylim=(-90, 60))
setup_axes_style(ax2c, xlabel="tiempo (ms)", ylabel="valor", title="Recuperación de h y n", xlim=(0, 55), ylim=(0, 1))
setup_axes_style(ax2d, xlabel="n", ylabel="h", title="Trayectoria en el plano (n, h)", xlim=(0, 1), ylim=(0, 1))

line2_V_full, = ax2a.plot([], [], lw=2, label="V(t)")
line2_I_full, = ax2a.plot([], [], lw=1.5, ls="--", label="I_ext")
vline2_p1 = ax2a.axvline(10.0, color="gray", ls="--", alpha=0.6)
vline2_p2 = ax2a.axvline(22.0, color="gray", ls="--", alpha=0.6)
text2 = ax2a.text(0.02, 0.95, "", transform=ax2a.transAxes, va="top")
ax2a.legend(loc="upper right")

line2_V_zoom, = ax2b.plot([], [], lw=2)
vline2_zoom = ax2b.axvline(22.0, color="gray", ls="--", alpha=0.6)

line2_h, = ax2c.plot([], [], lw=2, label="h (recuperación Na)")
line2_n, = ax2c.plot([], [], lw=2, label="n (activación K)")
vline2_rec = ax2c.axvline(22.0, color="gray", ls="--", alpha=0.6)
ax2c.legend(loc="best")

line2_traj, = ax2d.plot([], [], lw=2, label="trayectoria")
pt2_rest, = ax2d.plot([], [], marker="o", ms=6, linestyle="None", label="reposo")
pt2_second, = ax2d.plot([], [], marker="o", ms=7, linestyle="None", label="estado al 2º pulso")
ax2d.legend(loc="best")

plt.tight_layout()
plt.close(fig2)

def update_section_2(*_):
    p1_start = 10.0
    amp1 = amp1_slider_2.value
    dur1 = dur1_slider_2.value
    delay = delay_slider_2.value
    amp2 = amp2_slider_2.value
    dur2 = dur2_slider_2.value

    p2_start = p1_start + delay
    t_max = max(55.0, p2_start + 20.0)

    data = simulate_hh(
        pulses=[(p1_start, dur1, amp1), (p2_start, dur2, amp2)],
        t_max=t_max,
        dt=0.01
    )

    # picos del primer y segundo pulso
    t = data["t"]
    V = data["V"]

    mask1 = (t >= p1_start) & (t < p1_start + 15.0)
    mask2 = (t >= p2_start) & (t < p2_start + 15.0)

    peak1 = np.max(V[mask1]) if np.any(mask1) else np.nan
    peak2 = np.max(V[mask2]) if np.any(mask2) else np.nan

    if not np.isfinite(peak2) or peak2 < 0.0:
        estado = "refractario absoluto o estímulo insuficiente"
    elif peak2 < peak1 - 10.0:
        estado = "refractario relativo"
    else:
        estado = "recuperación casi completa"

    # tiempo completo
    line2_V_full.set_data(t, V)
    line2_I_full.set_data(t, data["I"] * 4.0 - 85.0)  # reescalado para verlo en el mismo panel
    vline2_p1.set_xdata([p1_start, p1_start])
    vline2_p2.set_xdata([p2_start, p2_start])

    # zoom
    line2_V_zoom.set_data(t, V)
    ax2b.set_xlim(max(0.0, p2_start - 8.0), p2_start + 15.0)
    vline2_zoom.set_xdata([p2_start, p2_start])

    # variables internas
    line2_h.set_data(t, data["h"])
    line2_n.set_data(t, data["n"])
    vline2_rec.set_xdata([p2_start, p2_start])

    # plano (n, h)
    line2_traj.set_data(data["n"], data["h"])
    pt2_rest.set_data([data["n"][0]], [data["h"][0]])
    idx_second = np.argmin(np.abs(t - p2_start))
    pt2_second.set_data([data["n"][idx_second]], [data["h"][idx_second]])

    text2.set_text(
        f"pico 1 = {peak1:.2f} mV\n"
        f"pico 2 = {peak2:.2f} mV\n"
        f"estado: {estado}"
    )

    fig2.canvas.draw_idle()

for w in (amp1_slider_2, dur1_slider_2, delay_slider_2, amp2_slider_2, dur2_slider_2):
    w.observe(update_section_2, names="value")

display(ui_2, fig2.canvas)
update_section_2()



### Lectura dinámica de esta sección

En esta sección conviene seguir dos variables:

- \(h\): disponibilidad del sodio.
- \(n\): activación del potasio.

Después del primer spike:

- \(h\) cae,
- \(n\) sube.

Eso hace que el sistema quede temporalmente en un estado menos excitable.  
En el plano \((n,h)\), el reposo está cerca de un punto con \(n\) bajo y \(h\) alto.  
El primer spike aleja al sistema de esa región y la recuperación corresponde al retorno gradual hacia ella.

Una regla mental útil:

- **\(h\) bajo** = menos capacidad de despolarización regenerativa.  
- **\(n\) alto** = más tendencia a repolarizar.  

Ambas empujan en contra de un segundo spike.



---

# 3. Rol de cada canal

## Interpretación física

Aquí puedes **encender y apagar** los canales de sodio y potasio para ver su papel de forma aislada.

### ¿Qué esperar?

- **Sin sodio**: desaparece la fase ascendente regenerativa.
- **Sin potasio**: el spike puede subir, pero la repolarización se vuelve débil o anómala.
- **Sin ambos**: queda una membrana casi pasiva, dominada por la fuga.
- **Sin fuga**: el comportamiento cambia menos durante el spike, pero el reposo efectivo y la relajación se alteran.

## Qué mirar aquí

- \(V(t)\): forma global del potencial.
- Corrientes iónicas: quién domina la dinámica.
- Conductancias efectivas: \(g_{Na}(t)\), \(g_K(t)\), \(g_L\).
- Curva cuasi-estática \(I_{\mathrm{ion}}(V)\): permite visualizar cómo cambian los balances de corriente al quitar canales.


In [ ]:

# =======================================
# Sección 3: Rol de cada canal
# =======================================

amp_slider_3 = widgets.FloatSlider(
    value=10.0, min=0.0, max=20.0, step=0.5,
    description="I", continuous_update=True
)
dur_slider_3 = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description="duración", continuous_update=True
)

na_toggle_3 = widgets.ToggleButtons(
    options=[("Na ON", 1.0), ("Na OFF", 0.0)],
    value=1.0, description=""
)
k_toggle_3 = widgets.ToggleButtons(
    options=[("K ON", 1.0), ("K OFF", 0.0)],
    value=1.0, description=""
)
l_toggle_3 = widgets.ToggleButtons(
    options=[("Leak ON", 1.0), ("Leak OFF", 0.0)],
    value=1.0, description=""
)

ui_3 = widgets.VBox([
    widgets.HTML("<b>Rol de cada canal</b>"),
    widgets.HBox([amp_slider_3, dur_slider_3]),
    widgets.HBox([
        widgets.VBox([widgets.HTML("Sodio"), na_toggle_3]),
        widgets.VBox([widgets.HTML("Potasio"), k_toggle_3]),
        widgets.VBox([widgets.HTML("Fuga"), l_toggle_3]),
    ])
])

fig3, axs3 = plt.subplots(2, 2, figsize=(11, 8))
ax3a, ax3b, ax3c, ax3d = axs3.ravel()

setup_axes_style(ax3a, xlabel="tiempo (ms)", ylabel="V (mV)", title="Potencial de membrana", xlim=(0, 40), ylim=(-90, 60))
setup_axes_style(ax3b, xlabel="tiempo (ms)", ylabel="corriente (uA/cm²)", title="Corrientes", xlim=(0, 40), ylim=(-250, 250))
setup_axes_style(ax3c, xlabel="tiempo (ms)", ylabel="conductancia (mS/cm²)", title="Conductancias efectivas", xlim=(0, 40), ylim=(0, 130))
setup_axes_style(ax3d, xlabel="V (mV)", ylabel="I_ion(V)", title="Curva cuasi-estática de corriente iónica", xlim=(-90, 60), ylim=(-50, 250))

line3_V, = ax3a.plot([], [], lw=2)
text3 = ax3a.text(0.02, 0.95, "", transform=ax3a.transAxes, va="top")

line3_INa, = ax3b.plot([], [], lw=2, label="I_Na")
line3_IK,  = ax3b.plot([], [], lw=2, label="I_K")
line3_IL,  = ax3b.plot([], [], lw=2, label="I_L")
line3_Iext,= ax3b.plot([], [], lw=2, ls="--", label="I_ext")
ax3b.legend(loc="upper right")

line3_gNa, = ax3c.plot([], [], lw=2, label="g_Na(t)")
line3_gK,  = ax3c.plot([], [], lw=2, label="g_K(t)")
line3_gL,  = ax3c.plot([], [], lw=2, label="g_L(t)")
ax3c.legend(loc="upper right")

V_grid_3 = np.linspace(-90, 60, 400)
line3_Iion, = ax3d.plot([], [], lw=2, label="I_ion(V)")
line3_Iext_h, = ax3d.plot([], [], lw=2, ls="--", label="I_ext")
ax3d.legend(loc="best")

plt.tight_layout()
plt.close(fig3)

def explain_channel_state(na_on, k_on, l_on):
    mensajes = []
    if na_on and k_on:
        mensajes.append("Na y K activos: comportamiento excitable normal.")
    if not na_on:
        mensajes.append("Sin Na: desaparece la subida regenerativa del spike.")
    if not k_on:
        mensajes.append("Sin K: la repolarización se debilita y el retorno al reposo cambia mucho.")
    if not na_on and not k_on:
        mensajes.append("Sin Na ni K: la membrana queda casi pasiva.")
    if not l_on:
        mensajes.append("Sin fuga: cambian el reposo efectivo y la relajación.")
    return "\n".join(mensajes)

def update_section_3(*_):
    amp = amp_slider_3.value
    dur = dur_slider_3.value
    na_on = float(na_toggle_3.value)
    k_on  = float(k_toggle_3.value)
    l_on  = float(l_toggle_3.value)

    data = simulate_hh(
        pulses=[(10.0, dur, amp)],
        t_max=40.0,
        dt=0.01,
        gNa_scale=na_on,
        gK_scale=k_on,
        gL_scale=l_on
    )

    line3_V.set_data(data["t"], data["V"])
    line3_INa.set_data(data["t"], data["INa"])
    line3_IK.set_data(data["t"], data["IK"])
    line3_IL.set_data(data["t"], data["IL"])
    line3_Iext.set_data(data["t"], data["I"])
    line3_gNa.set_data(data["t"], data["gNa_t"])
    line3_gK.set_data(data["t"], data["gK_t"])
    line3_gL.set_data(data["t"], data["gL_t"])

    Iion_curve = quasi_ionic_current(V_grid_3, data["params"])
    line3_Iion.set_data(V_grid_3, Iion_curve)
    line3_Iext_h.set_data(V_grid_3, np.full_like(V_grid_3, amp))

    summary = spike_summary(data, t_window_start=9.5)
    text3.set_text(
        f"V pico = {summary['peak_V']:.2f} mV\n"
        f"{explain_channel_state(bool(na_on), bool(k_on), bool(l_on))}"
    )

    fig3.canvas.draw_idle()

for w in (amp_slider_3, dur_slider_3, na_toggle_3, k_toggle_3, l_toggle_3):
    w.observe(update_section_3, names="value")

display(ui_3, fig3.canvas)
update_section_3()



### Lectura dinámica de esta sección

La curva \(I_{\mathrm{ion}}(V)\) es útil porque resume, de forma cuasi-estática, el balance entre las corrientes.

- Si el sodio está activo, puede aparecer una región donde el balance favorece la despolarización.
- Si quitas sodio, esa región desaparece y la excitabilidad cae.
- Si quitas potasio, el sistema pierde buena parte de su fuerza de retorno al reposo.
- Si solo queda la fuga, el sistema se comporta como una membrana casi lineal.

Aunque esta curva no reemplaza la dinámica completa en 4D, sí da una excelente intuición sobre **por qué** cambia el comportamiento al activar o desactivar canales.



---

# Apéndice: visualizaciones complementarias

Las siguientes celdas no son obligatorias para la parte interactiva principal, pero pueden servirte para estudiar mejor el modelo.


In [ ]:

# Curvas estacionarias y tiempos característicos de compuertas
V_aux = np.linspace(-100, 60, 500)

fig_aux, axs_aux = plt.subplots(1, 2, figsize=(11, 4))

axs_aux[0].plot(V_aux, m_inf(V_aux), label=r"$m_\infty(V)$", lw=2)
axs_aux[0].plot(V_aux, h_inf(V_aux), label=r"$h_\infty(V)$", lw=2)
axs_aux[0].plot(V_aux, n_inf(V_aux), label=r"$n_\infty(V)$", lw=2)
axs_aux[0].set_title("Curvas estacionarias")
axs_aux[0].set_xlabel("V (mV)")
axs_aux[0].set_ylabel("valor")
axs_aux[0].set_ylim(0, 1)
axs_aux[0].grid(True)
axs_aux[0].legend()

axs_aux[1].plot(V_aux, tau_m(V_aux), label=r"$\\tau_m(V)$", lw=2)
axs_aux[1].plot(V_aux, tau_h(V_aux), label=r"$\\tau_h(V)$", lw=2)
axs_aux[1].plot(V_aux, tau_n(V_aux), label=r"$\\tau_n(V)$", lw=2)
axs_aux[1].set_title("Tiempos característicos")
axs_aux[1].set_xlabel("V (mV)")
axs_aux[1].set_ylabel("ms")
axs_aux[1].grid(True)
axs_aux[1].legend()

plt.tight_layout()
plt.show()



## Ideas para extender este notebook

1. Comparar Euler vs. RK4.
2. Reducir el modelo a un sistema de menor dimensión.
3. Construir diagramas de bifurcación variando la corriente constante.
4. Estudiar trenes de spikes con corrientes sostenidas.
5. Pasar de una neurona aislada a un cable o a una red.

## Referencias breves

- A. L. Hodgkin y A. F. Huxley (1952), *A quantitative description of membrane current and its application to conduction and excitation in nerve*.  
- J. Keener y J. Sneyd, *Mathematical Physiology*.  
- E. M. Izhikevich, *Dynamical Systems in Neuroscience*.  
- C. Koch, *Biophysics of Computation*.
